# Case Study 4 — Three-class Amazon Review Sentiment

This notebook fine-tunes Hugging Face BERT and compares it with a real PyTorch BiLSTM. It intentionally contains no invented results: execute it to produce metrics.

In [ ]:
import re, random, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
df = pd.read_csv('Amazon_Product_Reviews.csv')
df['text'] = (df.review_headline.fillna('') + ' ' + df.review_body.fillna('')).str.replace(r'<[^>]+>', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
df = df[df.text.ne('')].copy()
df['label'] = np.select([df.star_rating <= 2, df.star_rating == 3, df.star_rating >= 4], [0,1,2]).astype(int)
label_names=['negative','neutral','positive']
df[['star_rating','label']].value_counts().sort_index(), len(df)


In [ ]:
train, temp = train_test_split(df[['text','label']], test_size=.30, stratify=df.label, random_state=SEED)
valid, test = train_test_split(temp, test_size=.50, stratify=temp.label, random_state=SEED)
print(len(train), len(valid), len(test))
def metric_dict(y_true, y_pred):
    p,r,f,_ = precision_recall_fscore_support(y_true,y_pred,average='macro',zero_division=0)
    return {'accuracy':accuracy_score(y_true,y_pred),'macro_precision':p,'macro_recall':r,'macro_f1':f}


In [ ]:
# LSTM baseline: vocabulary is fitted on training text only.
from collections import Counter
tokens=lambda s: re.findall(r"[a-z']+", s.lower())
vocab={'<pad>':0,'<unk>':1}
for word,_ in Counter(w for text in train.text for w in tokens(text)).most_common(30000): vocab[word]=len(vocab)
def encode(s,n=160): return ([vocab.get(w,1) for w in tokens(s)[:n]] + [0]*n)[:n]
class ReviewDataset(Dataset):
    def __init__(self, frame): self.x=torch.tensor([encode(t) for t in frame.text]); self.y=torch.tensor(frame.label.values)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.x[i],self.y[i]
class BiLSTM(nn.Module):
    def __init__(self): super().__init__(); self.emb=nn.Embedding(len(vocab),128,padding_idx=0); self.rnn=nn.LSTM(128,96,batch_first=True,bidirectional=True); self.out=nn.Linear(192,3)
    def forward(self,x): _,(h,_)=self.rnn(self.emb(x)); return self.out(torch.cat([h[-2],h[-1]],1))


In [ ]:
device='cuda' if torch.cuda.is_available() else 'cpu'; lstm=BiLSTM().to(device); opt=torch.optim.AdamW(lstm.parameters(),lr=2e-3); loss_fn=nn.CrossEntropyLoss()
for epoch in range(5):
    lstm.train()
    for x,y in DataLoader(ReviewDataset(train),batch_size=64,shuffle=True):
        opt.zero_grad(); loss=loss_fn(lstm(x.to(device)),y.to(device)); loss.backward(); opt.step()
    lstm.eval(); pred=[]
    with torch.no_grad():
        for x,_ in DataLoader(ReviewDataset(valid),batch_size=128): pred.extend(lstm(x.to(device)).argmax(1).cpu().tolist())
    print(epoch+1, metric_dict(valid.label, pred))
lstm.eval(); pred=[]
with torch.no_grad():
    for x,_ in DataLoader(ReviewDataset(test),batch_size=128): pred.extend(lstm(x.to(device)).argmax(1).cpu().tolist())
lstm_metrics=metric_dict(test.label,pred); print('LSTM test',lstm_metrics)


In [ ]:
tokenizer=AutoTokenizer.from_pretrained('bert-base-uncased')
class BertDataset(Dataset):
    def __init__(self,frame): self.e=tokenizer(frame.text.tolist(),truncation=True,padding=True,max_length=160); self.y=frame.label.tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return {**{k:torch.tensor(v[i]) for k,v in self.e.items()},'labels':torch.tensor(self.y[i])}
def bert_metrics(eval_pred):
    logits, labels=eval_pred; return metric_dict(labels,logits.argmax(1))
bert=AutoModelForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=3)
args=TrainingArguments(output_dir='bert_run',learning_rate=2e-5,per_device_train_batch_size=8,per_device_eval_batch_size=16,num_train_epochs=2,eval_strategy='epoch',save_strategy='no',report_to='none',seed=SEED)
trainer=Trainer(model=bert,args=args,train_dataset=BertDataset(train),eval_dataset=BertDataset(valid),compute_metrics=bert_metrics)
trainer.train(); bert_result=trainer.predict(BertDataset(test)); bert_pred=bert_result.predictions.argmax(1); bert_metrics_test=metric_dict(test.label,bert_pred)
print('BERT test',bert_metrics_test); print(classification_report(test.label,bert_pred,target_names=label_names,zero_division=0)); print(confusion_matrix(test.label,bert_pred))


In [ ]:
pd.DataFrame([lstm_metrics,bert_metrics_test],index=['BiLSTM','BERT']).round(4)
# Deployment: package tokenizer + model with model/version metadata; batch requests; validate length; log only IDs/metrics; monitor latency, confidence and class drift; route low-confidence predictions to human review.
